Step 0: Mounting Google Drive and creating the folders


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/doctalk'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/faiss_index', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/uploaded_docs', exist_ok=True)

print('Google Drive mounted and project folders ready.')
print(f'Project folder: {PROJECT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and project folders ready.
Project folder: /content/drive/MyDrive/doctalk


Step 1: Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pymupdf
!pip install -q sentence-transformers

print('All dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 28.0 MB/s eta 0:00:00
All dependencies installed.


Step 2: Uploading Documents


In [ ]:
from google.colab import files
import shutil

print('Please upload a PDF file...')
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f'{PROJECT_DIR}/uploaded_docs/{filename}'
    shutil.copy(filename, dest)
    PDF_PATH = dest
    print(f'Saved to Drive: {dest}')

print(f'\n Working with: {PDF_PATH}')

Please upload a PDF file...


Saving mars_lithograph  pdf.pdf to mars_lithograph  pdf.pdf
Saved to Drive: /content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf

 Working with: /content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf


Step 3: Loading and Extracting Text


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()

print(f'Loaded {len(pages)} pages from the PDF')
print(f'\n--- Preview of Page 1 (first 500 chars) ---')
print(pages[0].page_content[:500])
print(f'\n--- Metadata ---')
print(pages[0].metadata)

Loaded 2 pages from the PDF

--- Preview of Page 1 (first 500 chars) ---
National Aeronautics and Space Administration
National Aeronautics and Space Administration
300,000,000
900,000,000
1,500,000,000
2,100,000,000
2,700,000,000
3,300,000,000
3,900,000,000
4,500,000,000
5,100,000,000
5,700,000,000
kilometers
0
Mars
www.nasa.gov

--- Metadata ---
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260413181446', 'source': '/content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf', 'file_path': '/content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260413181446', 'page': 0}


Step 4: Splitting the text into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = splitter.split_documents(pages)

print(f'Split into {len(chunks)} chunks')
print(f'\n--- Example Chunk ---')
print(f'Content: {chunks[0].page_content}')
print(f'Metadata: {chunks[0].metadata}')
print(f'\n--- Next Chunk (notice the overlap) ---')
print(chunks[1].page_content[:200])

Split into 17 chunks

--- Example Chunk ---
Content: National Aeronautics and Space Administration
National Aeronautics and Space Administration
300,000,000
900,000,000
1,500,000,000
2,100,000,000
2,700,000,000
3,300,000,000
3,900,000,000
4,500,000,000
5,100,000,000
5,700,000,000
kilometers
0
Mars
www.nasa.gov
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260413181446', 'source': '/content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf', 'file_path': '/content/drive/MyDrive/doctalk/uploaded_docs/mars_lithograph  pdf.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260413181446', 'page': 0}

--- Next Chunk (notice the overlap) ---
Though details of Mars’ surface are difficult to see from Earth, 
telescope observations show seasonally changing features and 
white patches at the poles. For decades, people speculated that 
bright

Step 5: Generate Embeddings with HuggingFace

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

print('Loading embedding model.')

embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Quick test — embed a single sentence
test_vector = embeddings.embed_query('What is machine learning?')
print(f'\nEmbedding model loaded!')
print(f'Vector size: {len(test_vector)} dimensions')
print(f'First 5 values: {test_vector[:5]}')

Loading embedding model.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded!
Vector size: 384 dimensions
First 5 values: [-0.019954556599259377, 0.009878002107143402, 0.010249638929963112, 0.02955370768904686, 0.027186447754502296]


Step 6: Store the embeddings in the vector store

In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm

print(f'Embedding {len(chunks)} chunks and building FAISS index...\n')

# Embed chunks in batches with a progress bar
batch_size = 50
all_embeddings = []

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks", unit="batch"):
    batch = chunks[i:i + batch_size]
    batch_texts = [chunk.page_content for chunk in batch]
    batch_embeddings = embeddings.embed_documents(batch_texts)
    all_embeddings.extend(batch_embeddings)

print(f'\nEmbedded {len(all_embeddings)} chunks!')

# Build FAISS index from embeddings
print('Building FAISS index...')
texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
vectorstore = FAISS.from_embeddings(
    list(zip(texts, all_embeddings)),
    embeddings,
    metadatas=metadatas
)

# Save to Drive
FAISS_PATH = f'{PROJECT_DIR}/faiss_index'
vectorstore.save_local(FAISS_PATH)

print(f'FAISS index saved to Drive!')
print(f'Location: {FAISS_PATH}')

Embedding 17 chunks and building FAISS index...



Embedding chunks: 100%|██████████| 1/1 [00:01<00:00,  1.00s/batch]



Embedded 17 chunks!
Building FAISS index...
FAISS index saved to Drive!
Location: /content/drive/MyDrive/doctalk/faiss_index


Step 7: Query Test

In [ ]:
# Modify the query to something relevant to your document
query = "When did Viking 1 land on Mars?"

results = vectorstore.similarity_search(query, k=5)

print(f'Query: "{query}"')
print(f'\nTop {len(results)} most relevant chunks:\n')

for i, doc in enumerate(results):
    print(f'--- Chunk {i+1} (Page {doc.metadata.get("page", "?") + 1}) ---')
    print(doc.page_content)
    print()

Query: "When did Viking 1 land on Mars?"

Top 5 most relevant chunks:

--- Chunk 1 (Page 2) ---
–87 to –5 deg C (–125 to 23 deg F) 
Known Moons*	
2 
Rings	
0 
 
 
*As of July 2013.
SIGNIFICANT DATES
1877 — Asaph Hall discovers the two moons of Mars.  
1965 — NASA’s Mariner 4 sends back 22 photos of Mars, the 
world’s first close-up photos of a planet beyond Earth. 
1976 — Viking 1 and 2 land on the surface of Mars. 
1997 — Mars Pathfinder lands and dispatches Sojourner, the first 
wheeled rover to explore the surface of another planet.

--- Chunk 2 (Page 2) ---
Though details of Mars’ surface are difficult to see from Earth, 
telescope observations show seasonally changing features and 
white patches at the poles. For decades, people speculated that 
bright and dark areas on Mars were patches of vegetation, Mars 
was a likely place for advanced life forms, and water might exist 
in the polar caps. When the Mariner 4 spacecraft flew by Mars in 
1965, photographs of a bleak, cratered sur